# Walking a specification through the pipeline

For eyeballing one example end to end: the OSC source, the translation graph and the
constraints on each edge, the generated ASP instance, and the resulting plans as per-actor
traces.

Same thing is available from the shell:

```sh
python tests/run_speed_tests.py speed_start_end.osc -v
python tests/run_keep_tests.py keep_scenario_scope.osc -v
```

Run from the repo root with the venv active: `source venv/bin/activate && jupyter lab`.

In [ ]:
import os, sys, warnings

# Work from the repo root regardless of where the notebook was started.
while not os.path.exists("planning") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("tests"))
warnings.simplefilter("ignore")

from plan_checks import explain, states, speeds_of, lanes_of, goal_witnesses

print("cwd:", os.getcwd())

## Pick a specification

Anything under `tests/` or `experiments_src/`. A spec whose feature is not implemented yet
raises, which is the point: you see the source and the graph first, then exactly where it
stopped.

In [ ]:
SPEC = "tests/speed_start_end.osc"

plans = explain(SPEC, n=1)

## Poke at the plans

`explain` returns the plans, so the accessors from `plan_checks` work on them directly.

In [ ]:
m = plans[0]

print("goal witnesses:", goal_witnesses(m))
print("ego speeds    :", speeds_of(m, "ego"))
print("ego lanes     :", lanes_of(m, "ego"))
print()
for row in sorted(states(m), key=lambda r: (r[0], r[1])):
    print("actor=%-8s t=%-3s segment=%-4s lane=%-3s v_lon=%-3s v_lat=%s" % row)

## Check an invariant by hand

The same checkers the test runners use. Each returns `None` when the property holds, or a
string describing the first violation.

In [ ]:
from plan_checks import speed_at_start, speed_at_goal, speed_between, lane_equals

for name, check in [
    ("speed 5 at t=0", speed_at_start("ego", 5)),
    ("speed 10 at goal T1", speed_at_goal("ego", 1, 10)),
    ("speed within 0..12", speed_between("ego", 0, 12)),
    ("lane pinned to 2", lane_equals("ego", 2)),
]:
    result = check(m)
    print(f"{name:<22} {'holds' if result is None else 'VIOLATED: ' + result}")

## Compare two specifications

Useful for the pairs that are meant to be equivalent, such as a hand-written `keep` and the
modifier sugar that should desugar to it.

In [ ]:
def compare(spec_a, spec_b, actor="car1"):
    """Plan both and print their traces side by side."""
    out = {}
    for spec in (spec_a, spec_b):
        try:
            p = explain(spec, n=1, show_source=False, show_graph=False,
                        show_instance=False, show_plans=False, printer=lambda *_: None)
            out[spec] = (lanes_of(p[0], actor), speeds_of(p[0], actor))
        except Exception as exc:
            out[spec] = f"{type(exc).__name__}: {exc}"
    for spec, res in out.items():
        print(f"{os.path.basename(spec):<34} {res}")
    return out


# Both should give the same plan once the keep work lands.
_ = compare("tests/keep_lane_free.osc", "tests/keep_lane_modifier.osc")